In [1]:
!pip install unsloth

In [2]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig
from unsloth.chat_templates import get_chat_template

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

In [4]:
!unzip /content/qwen2.5-3b-cardio-chat.zip

Archive:  /content/qwen2.5-3b-cardio-chat.zip
   creating: qwen2.5-3b-cardio-chat/
  inflating: qwen2.5-3b-cardio-chat/adapter_config.json  
  inflating: qwen2.5-3b-cardio-chat/adapter_model.safetensors  
  inflating: qwen2.5-3b-cardio-chat/chat_template.jinja  
  inflating: qwen2.5-3b-cardio-chat/README.md  
  inflating: qwen2.5-3b-cardio-chat/tokenizer.json  
  inflating: qwen2.5-3b-cardio-chat/tokenizer_config.json  


In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/qwen2.5-3b-cardio-chat",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.6.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [6]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

Unsloth: Will map <|im_end|> to EOS = <|endoftext|>.


In [7]:
from huggingface_hub import login
login()

In [8]:
dataset = load_dataset("ai-galileo/clinical-notes-to-fhir", split="train")

README.md:   0%|          | 0.00/6.42k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/4.16M [00:00<?, ?B/s]

balanced_train_set.jsonl:   0%|          | 0.00/2.40M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/530 [00:00<?, ? examples/s]

In [9]:
dpo_dataset = dataset.filter(lambda x: x.get("valid") is False)

Filter:   0%|          | 0/530 [00:00<?, ? examples/s]

In [10]:
def format_dpo_pairs(example):

    user_prompt = str(example.get("note", ""))

    good_answer = "The clinical note has been reviewed and properly formatted without structural errors."
    bad_answer = "Error: " + str(example.get("validation_errors", "Invalid formatting detected."))

    prompt_chat = [
        {"role": "system", "content": "You are a helpful, empathetic, and expert medical professional."},
        {"role": "user", "content": user_prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(prompt_chat, tokenize=False, add_generation_prompt=True)

    return {
        "prompt": formatted_prompt,
        "chosen": good_answer + tokenizer.eos_token,
        "rejected": bad_answer + tokenizer.eos_token
    }

In [11]:
original_columns = dpo_dataset.column_names
dpo_dataset = dpo_dataset.map(format_dpo_pairs, remove_columns=original_columns)

Map:   0%|          | 0/223 [00:00<?, ? examples/s]

In [12]:
args = DPOConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 5e-6,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs_dpo",
    beta = 0.1,
)

trainer = DPOTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dpo_dataset,
    args = args,
)

Extracting prompt in train dataset (num_proc=6):   0%|          | 0/223 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/223 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/223 [00:00<?, ? examples/s]

In [13]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 223 | Num Epochs = 3 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
1,0.764536,1.447197,1.578690,0.250000,-0.131493,-56.697479,-148.675659,-0.095011,-0.177696
2,0.777457,1.335847,1.486117,0.375000,-0.150271,-57.765064,-127.126007,-0.038898,-0.334382
3,0.805509,1.359375,1.562718,0.125000,-0.203343,-57.244446,-126.826050,-0.009807,-0.180448
4,0.717639,1.398420,1.434579,0.375000,-0.036159,-56.134285,-114.441551,-0.055737,-0.039991
5,0.757559,1.480111,1.584249,0.500000,-0.104138,-55.454460,-117.586685,0.050192,-0.079524
6,0.655321,1.443533,1.357712,0.625000,0.085822,-54.820312,-120.370621,-0.099622,-0.007044
7,0.681284,1.485796,1.447540,0.375000,0.038256,-55.825325,-158.903564,-0.127712,-0.369090
8,0.586422,1.461810,1.218505,0.875000,0.243306,-56.001556,-132.412659,0.033622,-0.148349
9,0.487033,1.671068,1.161584,0.875000,0.509484,-54.842209,-144.501038,-0.115632,-0.353955
10,0.424268,1.715073,1.052926,1.000000,0.662147,-53.347713,-141.609360,-0.067052,-0.140195


Unsloth: Restored added_tokens_decoder metadata in outputs_dpo/checkpoint-60/tokenizer_config.json.


In [14]:
final_dpo_name = "qwen25-3b-cardiology-dpo-aligned"
model.save_pretrained(final_dpo_name)
tokenizer.save_pretrained(final_dpo_name)

Unsloth: Restored added_tokens_decoder metadata in qwen25-3b-cardiology-dpo-aligned/tokenizer_config.json.


('qwen25-3b-cardiology-dpo-aligned/tokenizer_config.json',
 'qwen25-3b-cardiology-dpo-aligned/chat_template.jinja',
 'qwen25-3b-cardiology-dpo-aligned/tokenizer.json')

In [15]:
!zip -r qwen25-3b-cardiology-dpo-aligned.zip /content/qwen25-3b-cardiology-dpo-aligned

  adding: content/qwen25-3b-cardiology-dpo-aligned/ (stored 0%)
  adding: content/qwen25-3b-cardiology-dpo-aligned/chat_template.jinja (deflated 59%)
  adding: content/qwen25-3b-cardiology-dpo-aligned/tokenizer_config.json (deflated 90%)
  adding: content/qwen25-3b-cardiology-dpo-aligned/README.md (deflated 65%)
  adding: content/qwen25-3b-cardiology-dpo-aligned/tokenizer.json (deflated 81%)
  adding: content/qwen25-3b-cardiology-dpo-aligned/adapter_config.json (deflated 59%)
  adding: content/qwen25-3b-cardiology-dpo-aligned/adapter_model.safetensors (deflated 8%)
